# Notebook 4: Data Preprocessing

This notebook covers comprehensive data preprocessing for the NYC Taxi Trip Duration dataset.

**Objectives:**
1. Handle missing values
2. Remove outliers
3. Encode categorical variables
4. Scale numerical features
5. Create train/test splits
6. Save preprocessed data

## 1. Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

# Add src to path
sys.path.append('../src')

# Import utilities
from utils.preprocessing import *
from utils.visualization import *
from config import *

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set random seed for reproducibility
np.random.seed(RANDOM_STATE)

print("Setup complete!")

Setup complete!


## 2. Load Engineered Dataset

In [2]:
# Load the engineered dataset
processed_dir = Path('../data/processed')
if (processed_dir / 'engineered_features.csv').exists():
    df = pd.read_csv(processed_dir / 'engineered_features.csv')
    print(f"Loaded engineered dataset from {processed_dir / 'engineered_features.csv'}")
else:
    # Load from raw data and apply feature engineering
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded raw dataset from {DATA_PATH}")
    
    # Apply basic feature engineering if needed
    from utils.feature_engineering import create_distance_features, create_temporal_features
    df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
    df = create_distance_features(df)
    df = create_temporal_features(df, 'pickup_datetime')
    
    # Select features for preprocessing
    selected_features = ['haversine_distance', 'manhattan_distance', 'hour', 'day_of_week', 
                        'is_rush_hour', 'is_weekend', 'average_speed_kmh', 'trip_duration']
    df = df[selected_features]
    
    print("Applied basic feature engineering")

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nTarget variable: trip_duration")

Loaded raw dataset from /Users/fharook/Documents/BTU TechHub/xai/nyc_taxi_regression/notebooks/../data/raw/NYC.csv


KeyError: "['average_speed_kmh'] not in index"

## 3. Data Quality Assessment

In [ ]:
print("Data Quality Assessment")
print("="*50)

# Missing values analysis
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_percent
})

print("Missing Values Analysis:")
print(missing_df)

# Data types
print("\n\nData Types:")
print(df.dtypes)

# Basic statistics
print("\n\nBasic Statistics:")
print(df.describe())

# Duplicate rows
duplicates = df.duplicated().sum()
print(f"\n\nDuplicate Rows: {duplicates} ({(duplicates/len(df))*100:.2f}%)")

## 4. Handle Missing Values

In [ ]:
print("Handling Missing Values")
print("="*30)

# Identify columns with missing values
missing_cols = df.columns[df.isnull().any()].tolist()
print(f"Columns with missing values: {missing_cols}")

if missing_cols:
    # Handle missing values based on data type and context
    for col in missing_cols:
        if df[col].dtype in ['int64', 'float64']:
            # For numerical columns, use median imputation
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)
            print(f"  {col}: Filled {df[col].isnull().sum()} missing values with median ({median_val:.2f})")
        else:
            # For categorical columns, use mode imputation
            mode_val = df[col].mode()[0]
            df[col].fillna(mode_val, inplace=True)
            print(f"  {col}: Filled {df[col].isnull().sum()} missing values with mode ({mode_val})")
else:
    print("No missing values found in the dataset.")

# Verify missing values are handled
remaining_missing = df.isnull().sum().sum()
print(f"\nRemaining missing values after imputation: {remaining_missing}")

## 5. Outlier Detection and Treatment

In [ ]:
print("Outlier Detection and Treatment")
print("="*40)

# Identify numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numerical_cols.remove('trip_duration')  # Keep target separate

print(f"Numerical columns for outlier analysis: {numerical_cols}")

# Visualize outliers
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

outlier_info = {}

for i, col in enumerate(numerical_cols):
    if i < len(axes):
        # Box plot
        axes[i].boxplot(df[col])
        axes[i].set_title(f'{col} - Outlier Detection')
        axes[i].set_ylabel(col)
        axes[i].grid(True, alpha=0.3)
        
        # Calculate IQR
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Count outliers
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        outlier_count = len(outliers)
        outlier_percentage = (outlier_count / len(df)) * 100
        
        outlier_info[col] = {
            'count': outlier_count,
            'percentage': outlier_percentage,
            'lower_bound': lower_bound,
            'upper_bound': upper_bound
        }
        
        axes[i].text(0.05, 0.95, f'Outliers: {outlier_count}\n({outlier_percentage:.1f}%)', 
                    transform=axes[i].transAxes, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Hide unused subplots
for i in range(len(numerical_cols), len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.savefig('../plots/eda/outlier_detection.png', dpi=300, bbox_inches='tight')
plt.show()

# Print outlier summary
print("\nOutlier Summary:")
for col, info in outlier_info.items():
    print(f"  {col:25s}: {info['count']:4d} outliers ({info['percentage']:5.1f}%)")

In [ ]:
# Handle outliers based on percentage
outlier_threshold = 5.0  # Remove features with more than 5% outliers
features_to_remove = []

print("\nOutlier Treatment:")
print("="*25)

for col, info in outlier_info.items():
    if info['percentage'] > outlier_threshold:
        features_to_remove.append(col)
        print(f"  {col}: {info['percentage']:.1f}% outliers - Feature will be removed")
    else:
        print(f"  {col}: {info['percentage']:.1f}% outliers - Outliers will be capped")
        
        # Cap outliers
        df[col] = np.clip(df[col], info['lower_bound'], info['upper_bound'])

# Remove features with high outlier percentage
if features_to_remove:
    df.drop(columns=features_to_remove, inplace=True)
    print(f"\nRemoved features: {features_to_remove}")

print(f"\nDataset shape after outlier treatment: {df.shape}")

## 6. Categorical Variable Encoding

In [ ]:
print("Categorical Variable Encoding")
print("="*35)

# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Categorical columns: {categorical_cols}")

if categorical_cols:
    # One-hot encoding for categorical variables
    df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    print(f"\nApplied one-hot encoding")
    print(f"Original shape: {df.shape}")
    print(f"Encoded shape: {df_encoded.shape}")
    print(f"Added {df_encoded.shape[1] - df.shape[1]} new columns")
    
    # Update dataframe
    df = df_encoded
else:
    print("No categorical variables found.")

# Check for boolean columns that need conversion
bool_cols = df.select_dtypes(include=['bool']).columns.tolist()
if bool_cols:
    print(f"\nConverting boolean columns: {bool_cols}")
    for col in bool_cols:
        df[col] = df[col].astype(int)
    print("Boolean conversion completed.")

## 7. Feature Scaling

In [ ]:
print("Feature Scaling")
print("="*20)

# Separate features and target
X = df.drop('trip_duration', axis=1)
y = df['trip_duration']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Apply scaling
scaler = StandardScalerCustom()
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

print("\nScaling completed.")
print(f"Original feature statistics:")
print(X.describe().loc[['mean', 'std']].T)
print(f"\nScaled feature statistics:")
print(X_scaled_df.describe().loc[['mean', 'std']].T)

In [ ]:
# Visualize scaling effect
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Select a few features to visualize
sample_features = X.columns[:6] if len(X.columns) >= 6 else X.columns

for i, feature in enumerate(sample_features):
    if i < len(axes):
        # Original distribution
        axes[i].hist(X[feature], bins=30, alpha=0.5, label='Original', color='blue')
        # Scaled distribution
        axes[i].hist(X_scaled_df[feature], bins=30, alpha=0.5, label='Scaled', color='red')
        axes[i].set_xlabel(feature)
        axes[i].set_ylabel('Frequency')
        axes[i].set_title(f'{feature} - Before/After Scaling')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)

# Hide unused subplots
for i in range(len(sample_features), len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.savefig('../plots/eda/scaling_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nScaling visualization saved.")

## 8. Train-Test Split

In [ ]:
print("Train-Test Split")
print("="*20)

# Create train-test split
X_train, X_test, y_train, y_test = train_test_split_custom(
    X_scaled_df, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")
print(f"Number of features: {X_train.shape[1]}")
print(f"\nTraining set target statistics:")
print(f"  Mean: {y_train.mean():.2f}")
print(f"  Std: {y_train.std():.2f}")
print(f"  Min: {y_train.min():.2f}")
print(f"  Max: {y_train.max():.2f}")

print(f"\nTest set target statistics:")
print(f"  Mean: {y_test.mean():.2f}")
print(f"  Std: {y_test.std():.2f}")
print(f"  Min: {y_test.min():.2f}")
print(f"  Max: {y_test.max():.2f}")

# Verify target distribution is similar
target_diff = abs(y_train.mean() - y_test.mean())
print(f"\nTarget mean difference: {target_diff:.2f}")
if target_diff < y_train.std() * 0.1:
    print("✓ Target distribution is well balanced between train and test sets")
else:
    print("⚠ Target distribution may be imbalanced")

## 9. Save Preprocessed Data

In [ ]:
# Create preprocessing results directory
preprocessed_dir = Path('../data/processed')
preprocessed_dir.mkdir(parents=True, exist_ok=True)

# Save preprocessed datasets
X_train.to_csv(preprocessed_dir / 'X_train.csv', index=False)
X_test.to_csv(preprocessed_dir / 'X_test.csv', index=False)
y_train.to_csv(preprocessed_dir / 'y_train.csv', index=False)
y_test.to_csv(preprocessed_dir / 'y_test.csv', index=False)

# Save feature names
feature_names = pd.DataFrame({'feature_name': X_train.columns})
feature_names.to_csv(preprocessed_dir / 'feature_names.csv', index=False)

# Save scaler parameters
scaler_params = pd.DataFrame({
    'feature': X_train.columns,
    'mean': scaler.mean_,
    'std': scaler.scale_
})
scaler_params.to_csv(preprocessed_dir / 'scaler_parameters.csv', index=False)

print(f"Preprocessed data saved to: {preprocessed_dir}")
print("Files created:")
print("- X_train.csv")
print("- X_test.csv")
print("- y_train.csv")
print("- y_test.csv")
print("- feature_names.csv")
print("- scaler_parameters.csv")
print("\nPlots saved to: ../plots/eda/")
print("- outlier_detection.png")
print("- scaling_comparison.png")

print(f"\n\nFinal preprocessed dataset shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test: {X_test.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  y_test: {y_test.shape}")
print(f"\nTotal features: {X_train.shape[1]}")
print(f"Total samples: {len(df)}")

## 10. Data Preprocessing Summary

### Data Preprocessing Results:

#### **Data Quality Issues Addressed:**
1. **Missing Values**: Handled using median/mode imputation
2. **Outliers**: Detected using IQR method, capped or removed
3. **Categorical Variables**: Encoded using one-hot encoding
4. **Feature Scaling**: Applied standardization (mean=0, std=1)
5. **Train-Test Split**: 80/20 split with stratification

#### **Preprocessing Steps Applied:**
1. **Missing Value Imputation**:
   - Numerical features: Median imputation
   - Categorical features: Mode imputation

2. **Outlier Treatment**:
   - IQR-based detection (1.5 * IQR rule)
   - Capping for moderate outliers (<5%)
   - Feature removal for extreme outliers (>5%)

3. **Categorical Encoding**:
   - One-hot encoding with drop_first=True
   - Boolean to integer conversion

4. **Feature Scaling**:
   - StandardScaler with custom implementation
   - Mean normalization and standard deviation scaling

5. **Data Splitting**:
   - 80% training, 20% testing
   - Random state for reproducibility
   - Balanced target distribution

#### **Final Dataset Statistics:**
- **Training samples**: {X_train.shape[0]:,}
- **Test samples**: {X_test.shape[0]:,}
- **Features**: {X_train.shape[1]}
- **Target variable**: trip_duration (seconds)

### Key Benefits:
1. **Improved Model Performance**: Scaled features prevent dominance
2. **Reduced Overfitting**: Outlier treatment improves generalization
3. **Better Convergence**: Standardized features speed up training
4. **Robust Evaluation**: Proper train-test split ensures reliable metrics